# Healthcare Operations SQL Validation Notebook

This notebook supports the validation expectations for AI-assisted SQL work. Use it before accepting an AI-generated query as correct. It helps you check schema fit, row counts, date ranges, status values, joins, filters, and expected outputs.

## Validation workflow

Use these cells in order when reviewing AI-generated SQL:

1. Confirm the database connection and available tables.
2. Inspect schemas and row counts.
3. Check fixed reporting dates and valid status values.
4. Check joins before trusting multi-table results.
5. Add filters one at a time.
6. Run expected-output checks or sanity checks.

In [ ]:
import sqlite3
import pandas as pd

DB_FILE = "healthcare_operations.db"
conn = sqlite3.connect(DB_FILE)
print(f"Connected to {DB_FILE}")

## 1. Table list check

Start by confirming that the database contains the tables your query expects. This helps catch prompts or SQL that assume the wrong database setup.

In [ ]:
tables = pd.read_sql("""
SELECT name AS table_name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""", conn)
tables

## 2. Schema inspection

Use the schema as the source of truth. Any table or column that does not appear here should be treated as a possible hallucination.

In [ ]:
for table_name in tables['table_name']:
    print(f"\n--- {table_name} ---")
    display(pd.read_sql(f"PRAGMA table_info({table_name});", conn))

## 3. Row counts

Check row counts before and after writing complex queries. If a source table has zero rows, a query that returns no rows may be behaving correctly.

In [ ]:
row_counts = []
for table_name in tables['table_name']:
    count = pd.read_sql(f"SELECT COUNT(*) AS row_count FROM {table_name};", conn).iloc[0, 0]
    row_counts.append({'table_name': table_name, 'row_count': count})
pd.DataFrame(row_counts)

## 4. Reporting context and min/max dates

Use the fixed reporting context instead of live-date logic such as `DATE('now')`. This prevents fictional historical datasets from returning empty or misleading results.

In [ ]:
pd.read_sql("SELECT * FROM reporting_context;", conn)

In [ ]:
pd.read_sql("SELECT * FROM dataset_date_range;", conn)

## 5. Valid status and category values

Check exact spelling and capitalization before writing filters. Small mismatches such as `no-show` vs. `no_show` or `canceled` vs. `Canceled` can change results.

In [ ]:
pd.read_sql("SELECT * FROM status_value_counts ORDER BY table_name, status_column, row_count DESC;", conn)

## 6. Join coverage checks

Check whether appointment records match valid patients, providers, and clinics before trusting a multi-table query.

In [ ]:
pd.read_sql("""
SELECT
    COUNT(*) AS total_appointments,
    COUNT(p.patient_id) AS appointments_with_matching_patient,
    COUNT(pr.provider_id) AS appointments_with_matching_provider,
    COUNT(c.clinic_id) AS appointments_with_matching_clinic
FROM appointments AS a
LEFT JOIN patients AS p
    ON a.patient_id = p.patient_id
LEFT JOIN providers AS pr
    ON a.provider_id = pr.provider_id
LEFT JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

## 7. Row-count-before-and-after join checks

These joins should preserve appointment-level rows when each appointment links to one patient, provider, and clinic.

In [ ]:
pd.read_sql("""
SELECT 'appointments only' AS query_step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'appointments + patients', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
UNION ALL
SELECT 'appointments + patients + providers + clinics', COUNT(*)
FROM appointments AS a
JOIN patients AS p
    ON a.patient_id = p.patient_id
JOIN providers AS pr
    ON a.provider_id = pr.provider_id
JOIN clinics AS c
    ON a.clinic_id = c.clinic_id;
""", conn)

## 8. Filter-by-filter debugging

Use exact status values from the database. In this dataset, missed appointments are stored as `no_show`.

In [ ]:
pd.read_sql("""
SELECT 'all appointments' AS step, COUNT(*) AS row_count FROM appointments
UNION ALL
SELECT 'last 3 months', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
UNION ALL
SELECT 'last 3 months and no_show', COUNT(*)
FROM appointments
WHERE appointment_date >= (SELECT recent_3_month_start FROM reporting_context)
  AND appointment_status = 'no_show';
""", conn)

## 9. Expected-output check: repeat no-show patients

This validates a common healthcare lab pattern: patients with repeated no-shows in the anchored recent period.

In [ ]:
repeat_no_show_patients = pd.read_sql("""
SELECT
    p.patient_id,
    p.patient_name,
    COUNT(a.appointment_id) AS no_show_count
FROM patients AS p
JOIN appointments AS a
    ON p.patient_id = a.patient_id
WHERE a.appointment_date >= (
    SELECT recent_3_month_start
    FROM reporting_context
)
  AND a.appointment_status = 'no_show'
GROUP BY p.patient_id, p.patient_name
HAVING COUNT(a.appointment_id) >= 2
ORDER BY no_show_count DESC, p.patient_name;
""", conn)

expected_columns = ['patient_id', 'patient_name', 'no_show_count']
assert list(repeat_no_show_patients.columns) == expected_columns
repeat_no_show_patients

In [ ]:
conn.close()
print("Database connection closed.")